# Pytorch Overview

In [97]:
import torch
import pandas as pd

In [98]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [99]:
import numpy as np

arr = np.array([[1,2,3],[4,5,6]])
arr

array([[1, 2, 3],
       [4, 5, 6]])

In [100]:
tensor = torch.Tensor([[1,2,3],[4,5,6]])
tensor

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [101]:
tensor * 5

tensor([[ 5., 10., 15.],
        [20., 25., 30.]])

Tensors are analogous to numpy arrays in many ways.

In [102]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [103]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [104]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [105]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [106]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [107]:
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x))

        return x

In [108]:
model = BCNet()

In [109]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [110]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss was {running_loss / len(train_loader)}')

Epoch 1: Loss was 0.6289152463277181
Epoch 2: Loss was 0.5054944515228271
Epoch 3: Loss was 0.37844120860099795
Epoch 4: Loss was 0.2633104741573334
Epoch 5: Loss was 0.1656365116437276
Epoch 6: Loss was 0.12503078530232112
Epoch 7: Loss was 0.10941680545608203
Epoch 8: Loss was 0.09058142527937889
Epoch 9: Loss was 0.07288269683097799
Epoch 10: Loss was 0.07144768635431925
Epoch 11: Loss was 0.06814585750301679
Epoch 12: Loss was 0.06607741564512253
Epoch 13: Loss was 0.058990847474584975
Epoch 14: Loss was 0.05319706834852696
Epoch 15: Loss was 0.06243838407099247
Epoch 16: Loss was 0.07075181429584822
Epoch 17: Loss was 0.049743361709018545
Epoch 18: Loss was 0.04977165646851063
Epoch 19: Loss was 0.05475935808693369
Epoch 20: Loss was 0.04530245171238979


In [111]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [112]:
accuracy

0.9824561476707458